# Implementing multiple turns

The rounds from the previous lesson, as a loop. Three functions:

| Function | Job |
|---|---|
| `run_conversation` | call Claude, store the reply, stop when `stop_reason != "tool_use"`, otherwise run the tools and send the results |
| `run_tools` | one reply can hold **several** `tool_use` blocks: run each, build a `tool_result` for each |
| `run_tool` | routing: tool name → Python function. A new tool means one more branch here, the loop stays the same |

Details from the lesson:

- the output goes into `content` as `json.dumps(tool_output)`, so any value (str, number, dict) becomes a string
- a failing tool still gets a `tool_result`, with `content` = the error text and `is_error: True`;
  Claude reads it and can retry with other arguments
- the text of every reply is printed, so you see what Claude says between tool calls

## Setup

In [1]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
from tools import (
    add_duration_to_datetime,
    add_duration_to_datetime_schema,
    get_current_datetime,
    get_current_datetime_schema,
)

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

## The loop

The course code. Two additions: `run_tool` raises on an unknown name instead of returning `None`, and `run_tools` prints each call so the rounds are visible.

In [2]:
import json

tools = [get_current_datetime_schema, add_duration_to_datetime_schema]


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:  # noqa: BLE001 - any tool error goes back to Claude
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        print(f"  [{tool_request.name}({tool_request.input}) -> {tool_result_block['content']}]")
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages


def print_history(messages):
    for number, message in enumerate(messages, start=1):
        content = message["content"]
        if isinstance(content, str):
            summary = repr(content)
        else:
            parts = []
            for block in content:
                if isinstance(block, dict):
                    parts.append(block["type"] + (" (error)" if block.get("is_error") else ""))
                else:
                    parts.append(block.type)
            summary = ", ".join(parts)
        print(f"{number}. {message['role']:9} {summary}")

## Two tools in one reply

Both values are independent, so Claude asks for them at once: one assistant message with two `tool_use` blocks, one user message with two `tool_result` blocks.

In [3]:
messages = []
add_user_message(messages, "What is the current time in HH:MM format? Also, what is the current time in SS format?")
run_conversation(messages)
print_history(messages)


  [get_current_datetime({'date_format': '%H:%M'}) -> "19:06"]
  [get_current_datetime({'date_format': '%S'}) -> "50"]


The current time is:
- **HH:MM format**: 19:06
- **SS format (seconds)**: 50

So the full current time is 19:06:50.
1. user      'What is the current time in HH:MM format? Also, what is the current time in SS format?'
2. assistant tool_use, tool_use
3. user      tool_result, tool_result
4. assistant text


## Two tools in a row

The second call needs the first result. The loop runs as many rounds as needed.

In [4]:
messages = []
add_user_message(messages, "What day is 103 days from today?")
run_conversation(messages)
print_history(messages)

I'll find out what day is 103 days from today.
  [get_current_datetime({'date_format': '%Y-%m-%d'}) -> "2026-10-29"]


Now I'll add 103 days to today's date:
  [add_duration_to_datetime({'datetime_str': '2026-10-29', 'input_format': '%Y-%m-%d', 'duration': 103, 'unit': 'days'}) -> "Tuesday, February 09, 2027 12:00:00 AM"]


103 days from today (October 29, 2026) is **Tuesday, February 9, 2027**.
1. user      'What day is 103 days from today?'
2. assistant text, tool_use
3. user      tool_result
4. assistant text, tool_use
5. user      tool_result
6. assistant text


## An error and a retry

`add_duration_to_datetime` parses the date with `input_format`, `%Y-%m-%d` by default. A first call
with `05.03.2026` and no format raises `ValueError`; `run_tools` catches it and sends the text back
with `is_error: True`. Claude reads it and calls again with a matching `input_format`.

The prompt forces the bad first call: on its own Claude reads the schema and avoids it. Asking for
3 "fortnights" (not a supported unit) did not fail either: the schema lists the units, so Claude
converted to 6 weeks before calling. Good descriptions prevent errors.

In [5]:
messages = []
add_user_message(
    messages,
    "Add 10 days to 05.03.2026 (March 5th). Call add_duration_to_datetime with "
    "datetime_str '05.03.2026' and do not pass input_format the first time.",
)
run_conversation(messages)
print_history(messages)


  [add_duration_to_datetime({'datetime_str': '05.03.2026', 'duration': 10, 'unit': 'days'}) -> Error: time data '05.03.2026' does not match format '%Y-%m-%d']


As expected, the function failed because the default input format is '%Y-%m-%d' (ISO format like '2026-03-05'), but the datetime string '05.03.2026' uses a different format (DD.MM.YYYY).

Now let me call it again with the correct input_format parameter:
  [add_duration_to_datetime({'datetime_str': '05.03.2026', 'duration': 10, 'unit': 'days', 'input_format': '%d.%m.%Y'}) -> "Sunday, March 15, 2026 12:00:00 AM"]


Perfect! Adding 10 days to **March 5th, 2026** gives us **Sunday, March 15, 2026 12:00:00 AM**.
1. user      "Add 10 days to 05.03.2026 (March 5th). Call add_duration_to_datetime with datetime_str '05.03.2026' and do not pass input_format the first time."
2. assistant tool_use
3. user      tool_result (error)
4. assistant text, tool_use
5. user      tool_result
6. assistant text


Next lesson: more tools, including `set_reminder`.